In [1]:
# module3_escalation_model.py

import torch
import torch.nn as nn
from transformers import AutoModel

class EscalationModel(nn.Module):
    def __init__(self, model_name="roberta-base", num_features=4):
        super().__init__()

        # Text encoder
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        # Feature projector (eval scores + sentiment)
        self.feature_proj = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Linear(64, 64)
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)  # {OK, ESCALATE}
        )

    def forward(self, input_ids, attention_mask, features):
        # Text embedding
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_embedding = outputs.last_hidden_state[:, 0]  # CLS token

        # Feature embedding
        feat_embedding = self.feature_proj(features)

        # Combine
        combined = torch.cat([cls_embedding, feat_embedding], dim=1)

        logits = self.classifier(combined)
        return logits

/home/souradeep/miniconda3/envs/TTA/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# module4_decision_engine.py

import torch
import torch.nn.functional as F

class DecisionEngine:
    def __init__(self, threshold=0.5):
        self.threshold = threshold

    def rule_based(self, features):
        """
        features = [faith, relevance, context_rel, sentiment]
        """
        faith, rel, ctx, sentiment = features

        if faith < 0.7:
            return 1  # escalate
        if rel < 0.7:
            return 1
        if sentiment < -0.5:
            return 1

        return 0

    def model_based(self, model, tokenizer, text, features, device):
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        ).to(device)

        features = torch.tensor(features).float().unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(
                inputs["input_ids"],
                inputs["attention_mask"],
                features
            )
            probs = F.softmax(logits, dim=-1)

        escalate_prob = probs[0][1].item()
        return escalate_prob

    def hybrid_decision(self, model, tokenizer, text, features, device):
        # Rule-based first (fast safety)
        rule_decision = self.rule_based(features)

        # Model-based confidence
        model_prob = self.model_based(model, tokenizer, text, features, device)

        # Combine
        final_score = 0.5 * rule_decision + 0.5 * model_prob

        if final_score > self.threshold:
            return "ESCALATE", final_score
        else:
            return "OK", final_score

In [3]:
# module5_logging.py

import json
import os
from datetime import datetime

class EscalationLogger:
    def __init__(self, log_path="logs/escalation_logs.jsonl"):
        self.log_path = log_path
        os.makedirs(os.path.dirname(log_path), exist_ok=True)

    def log(self, query, response, features, decision, score):
        log_entry = {
            "timestamp": str(datetime.now()),
            "query": query,
            "response": response,
            "features": {
                "faithfulness": features[0],
                "relevance": features[1],
                "context_relevance": features[2],
                "sentiment": features[3],
            },
            "decision": decision,
            "confidence": score
        }

        with open(self.log_path, "a") as f:
            f.write(json.dumps(log_entry) + "\n")

    def load_logs(self):
        data = []
        if not os.path.exists(self.log_path):
            return data

        with open(self.log_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        return data

In [8]:
# sentiment_vader.py

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

class SentimentAnalyzer:
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()

    def get_sentiment(self, text):
        scores = self.analyzer.polarity_scores(text)

        # compound score ∈ [-1, 1]
        sentiment = scores['compound']

        return sentiment

In [14]:
from transformers import AutoTokenizer
import torch

# from module3_escalation_model import EscalationModel
# from module4_decision_engine import DecisionEngine
# from module5_logging import EscalationLogger

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"

model = EscalationModel().to(device)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

decision_engine = DecisionEngine(threshold=0.5)
logger = EscalationLogger()
sentiment_model = SentimentAnalyzer()
# Example input
query = "Thank you for your kind service "
response = "Your welcome"
sentiment = sentiment_model.get_sentiment(query)
# Example evaluation outputs (from ragas)
features = [
    0.3,   # faithfulness
    0.4,   # relevance
    0.5,   # context relevance
    sentiment   # sentiment
]

text = query + " [SEP] " + response

# Decision
decision, score = decision_engine.hybrid_decision(
    model,
    tokenizer,
    text,
    features,
    device
)

# Logging
logger.log(query, response, features, decision, score)

print("Sentiment Score:", sentiment)
print("Decision:", decision)
print("Confidence:", score)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8648.77it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Sentiment Score: 0.7096
Decision: ESCALATE
Confidence: 0.7074206173419952
